# 04 — Nightlife Segment (Surprise Finding)
**Question:** Is Friday/Saturday night a distinct third market, and where should bikes be pre-positioned?

Key finding: Saturday midnight (210k rentals) = 2.3× Tuesday midnight. Avg night trip = 32.8 min vs fleet mean 21.9 min.

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT = 'm2-dataset-study'
engine = create_engine(f'bigquery://{PROJECT}/london_bicycles')
plt.rcParams['figure.dpi'] = 120

## 4.1 — Midnight volume by day of week

In [ ]:
sql = """
SELECT
    CASE EXTRACT(DAYOFWEEK FROM start_date)
        WHEN 1 THEN 'Sunday'   WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'  WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday' WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday' END AS day_name,
    EXTRACT(DAYOFWEEK FROM start_date) AS dow_num,
    SUM(CASE WHEN EXTRACT(HOUR FROM start_date) = 0 THEN 1 ELSE 0 END) AS midnight_rentals,
    SUM(CASE WHEN EXTRACT(HOUR FROM start_date) = 2 THEN 1 ELSE 0 END) AS twoam_rentals,
    ROUND(AVG(CASE WHEN EXTRACT(HOUR FROM start_date) BETWEEN 0 AND 4
              THEN duration_min END), 1) AS avg_nighttime_min
FROM `m2-dataset-study.london_bicycles.fact_rentals`
GROUP BY day_name, dow_num
ORDER BY dow_num
"""
df = pd.read_sql(sql, engine)

day_order = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
df['day_name'] = pd.Categorical(df['day_name'], categories=day_order, ordered=True)
df = df.sort_values('day_name')

fig, ax = plt.subplots(figsize=(11, 5))
colours = ['#6C3483' if d in ['Saturday','Sunday'] else '#BDC3C7' for d in df['day_name']]
bars = ax.bar(df['day_name'], df['midnight_rentals'], color=colours)
ax.set_title('Midnight Rentals by Day of Week — Saturday Night Dominates',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Total Midnight Rentals (all years)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{bar.get_height():,.0f}", ha='center', va='bottom', fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#6C3483', label='Nightlife days (Fri/Sat night)'),
                   Patch(color='#BDC3C7', label='Other days')], fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/04_midnight_by_dow.png', bbox_inches='tight')
plt.show()
print(df[['day_name','midnight_rentals','twoam_rentals','avg_nighttime_min']].to_string(index=False))

## 4.2 — Top nightlife departure stations

In [ ]:
sql2 = """
SELECT
    station_name,
    SUM(departures) AS total_nightlife_departures,
    ROUND(AVG(avg_duration_min), 1) AS avg_duration_min,
    latitude,
    longitude
FROM `m2-dataset-study.london_bicycles.mart_nightlife_demand`
WHERE station_name IS NOT NULL
GROUP BY station_name, latitude, longitude
ORDER BY total_nightlife_departures DESC
LIMIT 15
"""
top = pd.read_sql(sql2, engine)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top['station_name'], top['total_nightlife_departures'], color='#6C3483')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('Total Nightlife Departures (Fri/Sat 00:00–04:00, all years)')
ax.set_title('Top 15 Nightlife Departure Stations\n(pre-position bikes here before midnight)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/04_nightlife_top_stations.png', bbox_inches='tight')
plt.show()
print(top[['station_name','total_nightlife_departures','avg_duration_min']].to_string(index=False))

## 4.3 — Nightlife vs fleet average duration comparison

In [ ]:
dur_sql = """
SELECT
    trip_segment,
    ROUND(AVG(duration_min), 1) AS avg_min,
    COUNT(*) AS trips
FROM `m2-dataset-study.london_bicycles.fact_rentals`
WHERE NOT is_anomalous_duration
GROUP BY trip_segment
ORDER BY avg_min DESC
"""
dur = pd.read_sql(dur_sql, engine)

colours = {'nightlife': '#6C3483', 'leisure': '#F18F01', 'commute_am': '#2E86AB',
           'commute_pm': '#A23B72', 'daytime': '#95A5A6', 'night': '#4A4E69'}
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dur['trip_segment'], dur['avg_min'],
       color=[colours.get(s, '#888') for s in dur['trip_segment']])
ax.axhline(dur['avg_min'].mean(), color='red', linestyle='--', label=f'Overall mean')
ax.set_ylabel('Avg Duration (min)')
ax.set_title('Average Trip Duration by Segment', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/04_duration_by_segment.png', bbox_inches='tight')
plt.show()
print(dur.to_string(index=False))